In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
sys.path.append('../')
import igraph as ig

In [ ]:
freq_list = np.arange(1,41)
thresholds = np.arange(50, 91, 10)#[50, 65, 80]
plot = False

# Extract data and binarise connectivity matrices

In [ ]:
from data_extract import extract_wdpli
base_path_surv = "C:\\Users\\arthu\\Desktop\\Marzia\\marzia_data\\data_anonym_correct\\surv"
base_path_nsurv = "C:\\Users\\arthu\\Desktop\\Marzia\\marzia_data\\data_anonym_correct\\nonsurv"
dataframes_wdpli_surv, folders_surv = extract_wdpli(base_path_surv, surv=True)
dataframes_wdpli_nsurv, folders_nsurv = extract_wdpli(base_path_nsurv, surv=False)

print("Number of subjects in surv: ", len(folders_surv))
print("Number of subjects in non-surv: ", len(folders_nsurv))

In [ ]:
from data_extract import adj_percentile_thresholding
adj_mat_percentile_surv = {}
adj_mat_percentile_nsurv = {}
dict_id_pass_surv = []
dict_id_pass_nsurv = []

for p, patient in enumerate(folders_surv):
    for f, freq in enumerate(freq_list):
        for t, tau in enumerate(thresholds):
            data = np.nan_to_num(dataframes_wdpli_surv[f'{patient}_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{f+1}'].values)
            mat = adj_percentile_thresholding(data, tau)
            if mat.sum() == 0:
                dict_id_pass_surv.append((patient, freq, tau))
                print(f"Surviving Patient {patient} has a zero matrix at freq {freq} and threshold {tau}")
                pass
            else:
                adj_mat_percentile_surv[(patient, freq, tau)] = mat

for p, patient in enumerate(folders_nsurv):
    for f, freq in enumerate(freq_list):
        for t, tau in enumerate(thresholds):
            data = np.nan_to_num(dataframes_wdpli_nsurv[f'{patient}_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{f+1}'].values)
            mat = adj_percentile_thresholding(data, tau)
            if mat.sum() == 0:
                dict_id_pass_nsurv.append((patient, freq, tau))
                print(f"Non-surviving Patient {patient} has a zero matrix at freq {freq} and threshold {tau}")
                pass
            else:
                adj_mat_percentile_nsurv[(patient, freq, tau)] = mat

# Graphon estimation

In [ ]:
from loguru import logger
from pygraphon.estimators import USVT, NBDsmooth

estimators_percentile_surv = {}
estimators_percentile_nsurv = {}

for p, patient in enumerate(folders_surv):
    for f, freq in enumerate(freq_list):
        for t, tau in enumerate(thresholds):
            if (patient, freq, tau) in dict_id_pass_surv:
                pass
            else:
                estimator = NBDsmooth()
                estimator.fit(adj_mat_percentile_surv[(patient, freq, tau)])
                estimators_percentile_surv[(patient, freq, tau)] = estimator

for p, patient in enumerate(folders_nsurv):
    for f, freq in enumerate(freq_list):
        for t, tau in enumerate(thresholds):
            if (patient, freq, tau) in dict_id_pass_nsurv:
                pass
            else:
                estimator = NBDsmooth()
                estimator.fit(adj_mat_percentile_nsurv[(patient, freq, tau)])
                estimators_percentile_nsurv[(patient, freq, tau)] = estimator

In [ ]:
connectivity_surv = {}
connectivity_nsurv = {}

for f, freq in enumerate(freq_list):
    for t, tau in enumerate(thresholds):
        conn_surv = np.zeros((62,62))
        conn_nsurv = np.zeros((62,62))
        for p, patient in enumerate(folders_surv):
            if (patient, freq, tau) in dict_id_pass_surv:
                pass
            else:
                conn_surv += estimators_percentile_surv[(patient, freq, tau)].get_edge_connectivity()
        for p, patient in enumerate(folders_nsurv):
            if (patient, freq, tau) in dict_id_pass_nsurv:
                pass
            else:
                conn_nsurv += estimators_percentile_nsurv[(patient, freq, tau)].get_edge_connectivity()
        connectivity_surv[(freq, tau)] = conn_surv/len(folders_surv)
        connectivity_nsurv[(freq, tau)] = conn_nsurv/len(folders_nsurv)

#  Average graphon per frequency

## Estimation and performance

In [ ]:
from metrics import auc_ppv_score, t_test_topo_features, mannwhitneyu_topo_features
from electrodes_data import electrodes_experiments, electrodes_position_experiments
aucs = {}
ppvs = {}
npvs = {}
senss = {}
specs = {}
accurs = {}
for t, tau in enumerate(thresholds):
    # for each tau, store the (i,j)th element of the .get_edge_connectivity() matrix for the first dimension and second should be frequency
    for i in range(62):
        for j in range(62):
            surv = np.zeros((len(folders_surv), len(freq_list)))
            nsurv = np.zeros((len(folders_nsurv), len(freq_list)))
            for p, patient in enumerate(folders_surv):
                for f, freq in enumerate(freq_list):
                    if (patient, freq, tau) in dict_id_pass_surv:
                        pass
                    else:
                        surv[p, f] = estimators_percentile_surv[(patient, freq, tau)].get_edge_connectivity()[i, j]
            for p, patient in enumerate(folders_nsurv):
                for f, freq in enumerate(freq_list):
                    if (patient, freq, tau) in dict_id_pass_nsurv:
                        pass
                    else:
                        nsurv[p, f] = estimators_percentile_nsurv[(patient, freq, tau)].get_edge_connectivity()[i, j]
            auc, ppv, npv, sens, spec, accur = auc_ppv_score(surv, nsurv)
            for f, freq in enumerate(freq_list):
                aucs[(electrodes_experiments[i], electrodes_experiments[j], freq, tau)] = auc[f]
                ppvs[(electrodes_experiments[i], electrodes_experiments[j], freq, tau)] = ppv[f]
                npvs[(electrodes_experiments[i], electrodes_experiments[j], freq, tau)] = npv[f]
                senss[(electrodes_experiments[i], electrodes_experiments[j], freq, tau)] = sens[f]
                specs[(electrodes_experiments[i], electrodes_experiments[j], freq, tau)] = spec[f]
                accurs[(electrodes_experiments[i], electrodes_experiments[j], freq, tau)] = accur[f]


In [ ]:
#find in aucs, the top 5 values and their corresponding indices in the dictionnary
top_auc = sorted(aucs.items(), key=lambda x: x[1], reverse=True)[:10]
# print them with their corresponding ppv from ppvs
for i in range(10):
    print(top_auc[i], ppvs[top_auc[i][0]], npvs[top_auc[i][0]], senss[top_auc[i][0]], specs[top_auc[i][0]], accurs[top_auc[i][0]])

## Visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from electrodes_data import electrodes_experiments, electrodes_position_experiments
from sklearn.decomposition import PCA
from visualisation import plot_eeg_network

# Define the reduced electrode positions in 2d
reduced_electrode_positions = [(x,y) for x,y,_ in electrodes_position_experiments]

#pca = PCA(n_components=2)
#positions_2d = pca.fit_transform(electrodes_position_experiments)
#coordinates = {electrodes_experiments[i]: tuple(positions_2d[i]) for i in range(len(electrodes_experiments))}

freq = 2
tau = 90
plot_eeg_network(connectivity_nsurv[(freq, tau)], electrodes_experiments, reduced_electrode_positions, threshold = 0.1)

In [ ]:
group_diff_connectivity = {}
for f, freq in enumerate(freq_list):
    for t, tau in enumerate(thresholds):
        group_diff_connectivity[(freq, tau)] = (connectivity_surv[(freq, tau)] - connectivity_nsurv[(freq, tau)])/connectivity_surv[(freq, tau)]

for t, tau in enumerate(thresholds):
    fig, axs = plt.subplots(8, 5, figsize=(20, 30))
    for f, freq in enumerate(freq_list):
        min_val, max_val = np.nanmin(group_diff_connectivity[(freq, tau)]), np.nanmax(group_diff_connectivity[(freq, tau)])
        ax = axs[f//5, f%5]
        ax.imshow(group_diff_connectivity[(freq, tau)], cmap='jet', vmin=min_val, vmax=max_val)
        ax.set_title(f'Freq: {freq}Hz, Threshold: {tau}')
    if plot:
        plt.tight_layout()
        plt.savefig(f'C:\\Users\\arthu\\Documents\\Swittz-EPFL\\PhD\\Code\\Pictures\\Graphon\\group_diff_connectivity_threshold_{tau}.png', dpi=100)

# Slice of average graphon

In [ ]:
#do the smae but for .get_edge_connectivity() colmuns average
aucs_mean = {}
ppvs_mean = {}
npvs_mean = {}
senss_mean = {}
specs_mean = {}
accurs_mean = {}
for t, tau in enumerate(thresholds):
    # for each tau, store the (i,j)th element of the .get_edge_connectivity() matrix for the first dimension and second should be frequency
    for i in range(62):
        surv = np.zeros((len(folders_surv), len(freq_list)))
        nsurv = np.zeros((len(folders_nsurv), len(freq_list)))
        for p, patient in enumerate(folders_surv):
            for f, freq in enumerate(freq_list):
                if (patient, freq, tau) in dict_id_pass_surv:
                    pass
                else:
                    surv[p, f] = estimators_percentile_surv[(patient, freq, tau)].get_edge_connectivity()[i, :].mean()
        for p, patient in enumerate(folders_nsurv):
            for f, freq in enumerate(freq_list):
                if (patient, freq, tau) in dict_id_pass_nsurv:
                    pass
                else:
                    nsurv[p, f] = estimators_percentile_nsurv[(patient, freq, tau)].get_edge_connectivity()[i, :].mean()
        auc, ppv, npv, sens, spec, accur = auc_ppv_score(surv, nsurv)
        for f, freq in enumerate(freq_list):
            aucs_mean[(electrodes_experiments[i], freq, tau)] = auc[f]
            ppvs_mean[(electrodes_experiments[i], freq, tau)] = ppv[f]
            npvs_mean[(electrodes_experiments[i], freq, tau)] = npv[f]
            senss_mean[(electrodes_experiments[i], freq, tau)] = sens[f]
            specs_mean[(electrodes_experiments[i], freq, tau)] = spec[f]
            accurs_mean[(electrodes_experiments[i], freq, tau)] = accur[f]

In [ ]:
top_auc_mean = sorted(aucs_mean.items(), key=lambda x: x[1], reverse=True)[:10]
for i in range(10):
    print(top_auc_mean[i], ppvs_mean[top_auc_mean[i][0]], npvs_mean[top_auc_mean[i][0]], senss_mean[top_auc_mean[i][0]], specs_mean[top_auc_mean[i][0]], accurs_mean[top_auc_mean[i][0]])

# Cross frequency graphon

## Similarity across graphons

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

transitions = find_transitions(band_list)
thresholds_list = [50,90]
line_width = 0.5
line_style = 'solid'
line_color = 'red'
mse_mat_surv = {}
mse_mat_nsurv = {}
for t, tau in enumerate(thresholds):
    mse_mat_surv[tau] = np.zeros((len(freq_list), len(freq_list)))
    mse_mat_nsurv[tau] = np.zeros((len(freq_list), len(freq_list)))

    for f1, freq1 in enumerate(freq_list):
        for f2, freq2 in enumerate(freq_list):
            mse_mat_surv[tau][f1, f2] = np.mean((connectivity_surv[(freq1, tau)] - connectivity_surv[(freq2, tau)])**2)
            mse_mat_nsurv[tau][f1, f2] = np.mean((connectivity_nsurv[(freq1, tau)] - connectivity_nsurv[(freq2, tau)])**2)
    min_val = min(mse_mat_surv[tau].min(), mse_mat_nsurv[tau].min())
    max_val = max(mse_mat_surv[tau].max(), mse_mat_nsurv[tau].max())
    #plot the two matrices
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    im = axs[0].imshow(mse_mat_surv[tau], cmap='binary', vmin=min_val, vmax=max_val)
    im = axs[1].imshow(mse_mat_nsurv[tau], cmap='binary', vmin=min_val, vmax=max_val)
    for trans in transitions:
        axs[0].axvline(x=trans-0.5, color=line_color, linestyle=line_style, linewidth=line_width)
        axs[1].axvline(x=trans-0.5, color=line_color, linestyle=line_style, linewidth=line_width)
        axs[0].axhline(y=trans-0.5, color=line_color, linestyle=line_style, linewidth=line_width)
        axs[1].axhline(y=trans-0.5, color=line_color, linestyle=line_style, linewidth=line_width)
    axs[0].set_title('Survivors')
    axs[1].set_title('Non-survivors')
    axs[0].set_xlabel('Frequency')
    axs[0].set_ylabel('Frequency')
    axs[1].set_xlabel('Frequency')
    axs[1].set_ylabel('Frequency')
    # set x and y ticks to 4, 8, 12, 16, 30
    axs[0].set_xticks(transitions)
    axs[0].set_yticks(transitions)
    axs[1].set_xticks(transitions)
    axs[1].set_yticks(transitions)
    plt.colorbar(im, ax=axs)
    plt.savefig(f'C:\\Users\\arthu\\Documents\\Swittz-EPFL\\PhD\\Code\\Pictures\\Graphon\\mse_matrix_threshold_{tau}.png', dpi=300)
    plt.show()


## Hierarchical clustering

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import squareform

linkage_matrix_surv = {}
linkage_matrix_nsurv = {}
best_thresholds = {}
for t, tau in enumerate(thresholds):
    print(tau)
    # Linkage requires condensed distance matrices
    condensed_distance_matrix_surv = squareform(mse_mat_surv[tau], checks=False)
    condensed_distance_matrix_nsurv = squareform(mse_mat_nsurv[tau], checks=False)
    linkage_matrix_surv[tau] = linkage(condensed_distance_matrix_surv, 'average')
    linkage_matrix_nsurv[tau] = linkage(condensed_distance_matrix_nsurv, 'average')

    silhouette_scores = []
    # Use full symmetrical distance matrix for silhouette score
    full_distance_matrix = mse_mat_surv[tau]
    full_distance_matrix = (full_distance_matrix + full_distance_matrix.T) / 2
    np.fill_diagonal(full_distance_matrix, 0)

    thresholds_dendrogram = np.linspace(0, full_distance_matrix.max(), 100)
    for threshold_dendrogram in thresholds_dendrogram:
        labels = fcluster(linkage_matrix_surv[tau], threshold_dendrogram, criterion='distance')
        unique_labels = np.unique(labels)

        if 1 < len(unique_labels) < len(full_distance_matrix):
            score = silhouette_score(full_distance_matrix, labels, metric='precomputed')
            silhouette_scores.append((threshold_dendrogram, score))
    if silhouette_scores:
        best_threshold = max(silhouette_scores, key=lambda x: x[1])[0]
        best_thresholds[tau] = best_threshold
    else:
        print(f"No valid silhouette scores were calculated for {tau}.")
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    dendrogram(linkage_matrix_surv[tau], labels=freq_list, ax=axs[0])
    dendrogram(linkage_matrix_nsurv[tau], labels=freq_list, ax=axs[1])
    axs[0].set_title('Survivors')
    axs[1].set_title('Non-survivors')
    plt.tight_layout()
    plt.savefig(f'C:\\Users\\arthu\\Documents\\Swittz-EPFL\\PhD\\Code\\Pictures\\Graphon\\dendrogram_threshold_{tau}.png', dpi=300)
    plt.show()


In [ ]:
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import squareform

average_predictors_thresholded = {}
clusters_surv = {}
for t, tau in enumerate(thresholds):
    clusters_surv[tau] = fcluster(linkage_matrix_surv[tau], best_thresholds[tau], criterion='distance')
    clustered_predictors_surv = {i: [] for i in np.unique(clusters_surv[tau])}
    for idx, cluster_id in enumerate(clusters_surv[tau]):
        clustered_predictors_surv[cluster_id].append(connectivity_surv[(idx+1, tau)])
    average_predictors_thresholded[tau] = {k: np.mean(v, axis=0) for k, v in clustered_predictors_surv.items()}

In [ ]:
from metrics import auc_ppv_score, t_test_topo_features, mannwhitneyu_topo_features
aucs = {}
ppvs = {}
for t, tau in enumerate(thresholds):
    # for each tau, store the (i,j)th element of the .get_edge_connectivity() matrix for the first dimension and second should be frequency
    for i in range(62):
        for j in range(62):
            surv = np.zeros((len(folders_surv), len(np.unique(clusters_surv[tau]))))
            nsurv = np.zeros((len(folders_nsurv), len(np.unique(clusters_surv[tau]))))
            for p, patient in enumerate(folders_surv):
                for f, f_clust in enumerate(clusters_surv[tau]):
                    if (patient, f+1, tau) in dict_id_pass_surv:
                        pass
                    else:
                        surv[p, f_clust-1] += estimators_percentile_surv[(patient, f+1, tau)].get_edge_connectivity()[i, j]
            for p, patient in enumerate(folders_nsurv):
                for f, f_clust in enumerate(clusters_surv[tau]):
                    if (patient, f+1, tau) in dict_id_pass_nsurv:
                        pass
                    else:
                        nsurv[p, f_clust-1] = estimators_percentile_nsurv[(patient, f+1, tau)].get_edge_connectivity()[i, j]
            auc, ppv = auc_ppv_score(surv, nsurv, freq_list=np.unique(clusters_surv[tau]))
            for f, f_clust in enumerate(np.unique(clusters_surv[tau])):
                aucs[(i+1, j+1, f_clust, tau)] = auc[f]
                ppvs[(i+1, j+1, f_clust, tau)] = ppv[f]


In [ ]:
#find in aucs, the top 5 values and their corresponding indices in the dictionnary
top_auc = sorted(aucs.items(), key=lambda x: x[1], reverse=True)[:10]
# print them with their corresponding ppv from ppvs
for i in range(10):
    print(top_auc[i], ppvs[top_auc[i][0]])

## Clustering based on bands

In [ ]:
from data_extract import get_band
band_list = [get_band(freq) for freq in freq_list]
band_names = np.unique(band_list)

# average the connectivity matrices over the band groups
graphon_band_surv = {}
graphon_band_nsurv = {}
patient_band_surv = {}
patient_band_nsurv = {}
for b, band in enumerate(band_names):
    # find where the band is in the freq_list
    band_indices = [i for i, x in enumerate(band_list) if x == band]    

    for t, tau in enumerate(thresholds):
        graphon_band_surv[band, tau] = np.zeros((62,62))
        graphon_band_nsurv[band, tau] = np.zeros((62,62))

        for p, patient in enumerate(folders_surv):
            patient_band_surv[(patient, band, tau)] = np.zeros((62,62))
        for p, patient in enumerate(folders_nsurv):
            patient_band_nsurv[(patient, band, tau)] = np.zeros((62,62))

        for f in band_indices:
            graphon_band_surv[band, tau] += connectivity_surv[(freq_list[f], tau)]
            graphon_band_nsurv[band, tau] += connectivity_nsurv[(freq_list[f], tau)]
            for p, patient in enumerate(folders_surv):
                if (patient, freq_list[f], tau) in dict_id_pass_surv:
                    pass
                else:
                    patient_band_surv[(patient, band, tau)] += estimators_percentile_surv[(patient, freq_list[f], tau)].get_edge_connectivity()
            for p, patient in enumerate(folders_nsurv):
                if (patient, freq_list[f], tau) in dict_id_pass_nsurv:
                    pass
                else:
                    patient_band_nsurv[(patient, band, tau)] += estimators_percentile_nsurv[(patient, freq_list[f], tau)].get_edge_connectivity()

        graphon_band_surv[band, tau] /= len(band_indices)
        graphon_band_nsurv[band, tau] /= len(band_indices)
        for p, patient in enumerate(folders_surv):
            patient_band_surv[(patient, band, tau)] /= len(band_indices)
        for p, patient in enumerate(folders_nsurv):
            patient_band_nsurv[(patient, band, tau)] /= len(band_indices)
    
    # plot the graphons
    if plot:
        fig, axs = plt.subplots(1, 2, figsize=(10, 5))
        min_val, max_val = np.nanmin(graphon_band_surv[band, tau]), np.nanmax(graphon_band_surv[band, tau])
        min_val = min(min_val, np.nanmin(graphon_band_nsurv[band, tau]))
        max_val = max(max_val, np.nanmax(graphon_band_nsurv[band, tau]))
        im = axs[0].imshow(graphon_band_surv[band, tau], cmap='jet', vmin=min_val, vmax=max_val)
        im = axs[1].imshow(graphon_band_nsurv[band, tau], cmap='jet', vmin=min_val, vmax=max_val)
        # add colorbar
        plt.colorbar(im, ax=axs)
        axs[0].set_title('Survivors')
        axs[1].set_title('Non-survivors')
        # add overhead title
        fig.suptitle(f'Band: {band}')
        plt.savefig(f'C:\\Users\\arthu\\Documents\\Swittz-EPFL\\PhD\\Code\\Pictures\\Graphon\\graphon_band_{band}.png', dpi=300)
        plt.show()

In [ ]:
from visualisation import plot_dual_eeg_network
from electrodes_data import electrodes_experiments, electrodes_position_experiments
reduced_electrode_positions = [(x,y) for x,y,_ in electrodes_position_experiments]

for b, band in enumerate(band_names):
    folder_path = f'C:\\Users\\arthu\\Documents\\Swittz-EPFL\\PhD\\Code\\Pictures\\Graphon\\{band}\\'
    print(folder_path)
    for t, tau in enumerate(thresholds):
        plot_dual_eeg_network(graphon_band_surv[(band, tau)],
                         graphon_band_nsurv[(band, tau)],
                         electrodes_experiments,
                         reduced_electrode_positions,
                         threshold = tau,
                         folder_path=folder_path,)

In [ ]:
from metrics import auc_ppv_score, t_test_topo_features, mannwhitneyu_topo_features
aucs = {}
ppvs = {}

for t, tau in enumerate(thresholds):
    # for each tau, store the (i,j)th element of the .get_edge_connectivity() matrix for the first dimension and second should be frequency
    for i in range(62):
        for j in range(62):
            surv = np.zeros((len(folders_surv),len(band_names)))
            nsurv = np.zeros((len(folders_nsurv),len(band_names)))
            for b, band in enumerate(band_names):
                band_indices = [k for k, x in enumerate(band_list) if x == band]
                for p, patient in enumerate(folders_surv):
                    for f in band_indices:
                        if (patient, freq_list[f], tau) in dict_id_pass_surv:
                            pass
                        else:
                            surv[p,b] = patient_band_surv[(patient, band, tau)][i, j]
                for p, patient in enumerate(folders_nsurv):
                    for f in band_indices:
                        if (patient, freq_list[f], tau) in dict_id_pass_nsurv:
                            pass
                        else:
                            nsurv[p, b] = patient_band_nsurv[(patient, band, tau)][i, j]
                aucs[(i+1, j+1, band, tau)], ppvs[(i+1, j+1, band, tau)] = auc_ppv_score(surv, nsurv, freq_list=band_names)

In [ ]:
#find in aucs, the top 5 values and their corresponding indices in the dictionnary
top_auc = sorted(aucs.items(), key=lambda x: x[1], reverse=True)[:10]
# print them with their corresponding ppv from ppvs
for i in range(10):
    print(top_auc[i], ppvs[top_auc[i][0]])